In [2]:
import time, statistics
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DTYPE = torch.float32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW = 48

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()

gen_eos = model.generation_config.eos_token_id
EOS_IDS = set(gen_eos if isinstance(gen_eos, list) else [gen_eos])
GREEDY = dict(do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0)

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        
print("pad token:", repr(tokenizer.pad_token), tokenizer.pad_token_id, "| EOS ids:", EOS_IDS,
      "| pad is also an EOS id:", tokenizer.pad_token_id in EOS_IDS)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 160.18it/s]


pad token: '<|endoftext|>' 151643 | EOS ids: {151643, 151645} | pad is also an EOS id: True


In [3]:
QUESTIONS = [
    "Answer with one word: what colour is the sky on a clear day?",
    "What is 12 times 7? Reply with only the number.",
    "Write three sentences about why the ocean is salty.",
    "List five fruits, comma separated.",
    "Explain in detail how a refrigerator keeps food cold.",
    "Say hello.",
    "Give a short definition of photosynthesis.",
    "Write a short paragraph about the history of the bicycle.",
    "What is the capital of Japan? One word.",
    "Describe how rain forms, step by step.",
    "Translate 'good morning' into French.",
    "Write a haiku about coffee.",
    "Explain what a database index is and why it speeds up queries.",
    "Name the planet closest to the sun.",
    "Give three tips for writing clean Python code.",
    "Summarise the water cycle in two sentences.",
]

def to_prompt(q):
    return tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)

PROMPTS = [to_prompt(q) for q in QUESTIONS]
print("prompt lengths (tokens):", [len(tokenizer(p, add_special_tokens=False)["input_ids"]) for p in PROMPTS])

prompt lengths (tokens): [44, 44, 39, 36, 40, 32, 37, 40, 39, 38, 37, 36, 43, 37, 38, 39]


##### See padding and the attention mask

In [4]:
demo = PROMPTS[:3]
for side in ["left", "right"]:
    tokenizer.padding_side = side
    enc = tokenizer(demo, return_tensors="pt", padding=True, add_special_tokens=False)
    print(f"===== padding_side = {side!r}  -> tensor shape {tuple(enc.input_ids.shape)} =====")
    
    for i in range(len(demo)):
        ids = enc.input_ids[i].tolist()
        mask = enc.attention_mask[i].tolist()
        print(f"row {i}: first 4 ids {ids[:4]} ... last 4 ids {ids[-4:]}")
        print(f"       first 4 mask {mask[:4]} ... last 4 mask {mask[-4:]}  (pad tokens: {mask.count(0)})")
    
tokenizer.padding_side = "left"
print("\nLeft padding puts pads at the START, so every row's LAST position is a real token —")
print("that is where generate() reads the next-token logits and appends the new token.")

===== padding_side = 'left'  -> tensor shape (3, 44) =====
row 0: first 4 ids [151644, 8948, 198, 2610] ... last 4 ids [198, 151644, 77091, 198]
       first 4 mask [1, 1, 1, 1] ... last 4 mask [1, 1, 1, 1]  (pad tokens: 0)
row 1: first 4 ids [151644, 8948, 198, 2610] ... last 4 ids [198, 151644, 77091, 198]
       first 4 mask [1, 1, 1, 1] ... last 4 mask [1, 1, 1, 1]  (pad tokens: 0)
row 2: first 4 ids [151643, 151643, 151643, 151643] ... last 4 ids [198, 151644, 77091, 198]
       first 4 mask [0, 0, 0, 0] ... last 4 mask [1, 1, 1, 1]  (pad tokens: 5)
===== padding_side = 'right'  -> tensor shape (3, 44) =====
row 0: first 4 ids [151644, 8948, 198, 2610] ... last 4 ids [198, 151644, 77091, 198]
       first 4 mask [1, 1, 1, 1] ... last 4 mask [1, 1, 1, 1]  (pad tokens: 0)
row 1: first 4 ids [151644, 8948, 198, 2610] ... last 4 ids [198, 151644, 77091, 198]
       first 4 mask [1, 1, 1, 1] ... last 4 mask [1, 1, 1, 1]  (pad tokens: 0)
row 2: first 4 ids [151644, 8948, 198, 2610] ... 

In [5]:
def first_eos_cut(row):
    for i, t in enumerate(row):
        if t in EOS_IDS:
            return i + 1          # keep the EOS itself in the count
    return len(row)

@torch.inference_mode()
def gen_batch(texts, max_new=MAX_NEW, side="left", pass_mask=True):
    tokenizer.padding_side = side
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
    kwargs = dict(input_ids=enc.input_ids, max_new_tokens=max_new,
                  pad_token_id=tokenizer.pad_token_id, **GREEDY)
    if pass_mask:
        kwargs["attention_mask"] = enc.attention_mask
    
    sync()
    t0 = time.perf_counter()
    
    out = model.generate(**kwargs)

    sync()
    secs = time.perf_counter() - t0
    new = out[:, enc.input_ids.shape[1]:]  # drop the (padded) prompt part
    
    rows = new.tolist()
    
    useful = [first_eos_cut(r) for r in rows]  # real tokens per row (until its EOS)
    texts_out = [tokenizer.decode(r[:n], skip_special_tokens=True) for r,n in zip(rows, useful)]
    return {"texts": texts_out, "useful": useful, "steps": new.shape[1], "secs": secs,
            "prompt_pad": int((enc.attention_mask == 0).sum())}
    
_ = gen_batch(PROMPTS[:2], max_new=4)   # warm-up

subset = PROMPTS[:4]
solo = [gen_batch([p])["texts"][0] for p in subset]
left = gen_batch(subset, side="left")["texts"]
right = gen_batch(subset, side="right")["texts"]
nomask = gen_batch(subset, side="left", pass_mask=False)["texts"]

def report(name, outs):
    same = [a == b for a, b in zip(outs, solo)]
    print(f"{name:28s} matches one-at-a-time: {sum(same)}/{len(same)}  {same}")

report("LEFT pad + attention_mask", left)
report("RIGHT pad + attention_mask", right)
report("LEFT pad, NO attention_mask", nomask)
print("\nexample (row 0):")
print("  solo :", repr(solo[0][:80]))
print("  right:", repr(right[0][:80]))
print("  nomask:", repr(nomask[0][:80]))


[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


LEFT pad + attention_mask    matches one-at-a-time: 4/4  [True, True, True, True]
RIGHT pad + attention_mask   matches one-at-a-time: 2/4  [True, True, False, False]
LEFT pad, NO attention_mask  matches one-at-a-time: 2/4  [True, True, False, False]

example (row 0):
  solo : 'Blue'
  right: 'Blue'
  nomask: 'Blue'


##### Throughput vs latency at batch sizes 1, 2, 4, 8 and 16

In [9]:
def run_all(prompts, batch_size):
    clock = 0.0
    latencies = []
    useful_total = 0
    slots_total = 0
    prompt_pad_total = 0
    
    for s in range(0, len(prompts), batch_size):
        r = gen_batch(prompts[s:s + batch_size])
        clock += r["secs"]
        latencies += [clock] * len(r["useful"])       # all requests arrived at t=0; batches run one after another
        useful_total += sum(r["useful"])
        slots_total += len(r["useful"]) * r["steps"]   # rows x decode steps actually computed
        prompt_pad_total += r["prompt_pad"]
        
    return {"bs": batch_size, "total_s": clock, "useful_tokens": useful_total,
            "tok_per_s": useful_total / clock, "mean_latency_s": statistics.mean(latencies),
            "max_latency_s": max(latencies), "decode_waste": 1 - useful_total / slots_total,
            "prompt_pad_tokens": prompt_pad_total}
    
BATCH_SIZES = [1, 2, 4, 8, 16]
results = []
print(f"{'batch':>5} | {'total s':>7} | {'useful tok/s':>12} | {'mean latency s':>14} | "
      f"{'max latency s':>13} | {'decode waste':>12} | {'prompt pads':>11}")

for bs in BATCH_SIZES:
    r = run_all(PROMPTS, bs)
    results.append(r)
    print(f"{r['bs']:>5} | {r['total_s']:>7.1f} | {r['tok_per_s']:>12.1f} | {r['mean_latency_s']:>14.1f} | "
          f"{r['max_latency_s']:>13.1f} | {r['decode_waste']:>11.0%} | {r['prompt_pad_tokens']:>11}")


batch | total s | useful tok/s | mean latency s | max latency s | decode waste | prompt pads
    1 |    40.1 |         11.3 |           17.5 |          40.1 |          0% |           0
    2 |    40.1 |         11.3 |           22.9 |          40.1 |         27% |          23
    4 |    22.8 |         19.9 |           14.3 |          22.8 |         41% |          45
    8 |    16.7 |         27.2 |           12.5 |          16.7 |         41% |          77
   16 |    11.8 |         38.5 |           11.8 |          11.8 |         41% |          85


##### Head-of-line blocking

In [11]:
short_p = to_prompt("What is 2 + 2? Reply with only the number.")
long_p = to_prompt("Write a detailed paragraph about the history of computers.")

alone = gen_batch([short_p], max_new=96)
together = gen_batch([short_p, long_p], max_new=96)

print(f"short request ALONE        : {alone['useful'][0]} tokens, returned after {alone['secs']:.2f}s")
print(f"short request WITH long one: {together['useful'][0]} tokens, returned after {together['secs']:.2f}s "
      f"(the batch ran {together['steps']} steps for the long row: {together['useful'][1]} tokens)")
print(f"same answer both ways: {alone['texts'][0] == together['texts'][0]}  -> {alone['texts'][0]!r}")
print(f"\nThe short user waited {together['secs'] / alone['secs']:.1f}x longer only because of who it was batched with.")


short request ALONE        : 2 tokens, returned after 0.26s
short request WITH long one: 2 tokens, returned after 7.28s (the batch ran 96 steps for the long row: 96 tokens)
same answer both ways: True  -> '4'

The short user waited 28.3x longer only because of who it was batched with.


##### Sorting by prompt length

In [12]:
lengths = [len(tokenizer(p, add_special_tokens=False)["input_ids"]) for p in PROMPTS]
order = sorted(range(len(PROMPTS)), key=lambda i: lengths[i])
sorted_prompts = [PROMPTS[i] for i in order]

def prompt_pads(prompts, bs):
    total = 0
    for s in range(0, len(prompts), bs):
        chunk = [lengths_map[p] for p in prompts[s:s + bs]]
        total += sum(max(chunk) - x for x in chunk)
    return total

lengths_map = dict(zip(PROMPTS, lengths))
BS = 4
print(f"prompt pad tokens with batch size {BS}: original order {prompt_pads(PROMPTS, BS)}, "
      f"sorted by length {prompt_pads(sorted_prompts, BS)}")
r_orig = run_all(PROMPTS, BS)
r_sorted = run_all(sorted_prompts, BS)
print(f"original order: {r_orig['total_s']:.1f}s, decode waste {r_orig['decode_waste']:.0%}")
print(f"sorted order  : {r_sorted['total_s']:.1f}s, decode waste {r_sorted['decode_waste']:.0%}")
print("\nSorting removes PROMPT padding. It cannot remove DECODE waste, because answer")
print("lengths are unknown until the model finishes — that is what continuous batching fixes.")

prompt pad tokens with batch size 4: original order 45, sorted by length 17
original order: 29.6s, decode waste 41%
sorted order  : 26.4s, decode waste 41%

Sorting removes PROMPT padding. It cannot remove DECODE waste, because answer
lengths are unknown until the model finishes — that is what continuous batching fixes.


In [13]:
best = max(results, key=lambda r: r["tok_per_s"])
one = results[0]
print(f"Model {MODEL_ID} | {DTYPE} | {DEVICE} | {len(PROMPTS)} prompts | max_new_tokens={MAX_NEW}")
print(f"batch 1  : {one['tok_per_s']:.1f} useful tok/s, mean latency {one['mean_latency_s']:.1f}s")
print(f"best (bs={best['bs']}): {best['tok_per_s']:.1f} useful tok/s, mean latency {best['mean_latency_s']:.1f}s, "
      f"decode waste {best['decode_waste']:.0%}")
print(f"throughput gain from batching: {best['tok_per_s'] / one['tok_per_s']:.2f}x")

Model Qwen/Qwen2.5-0.5B-Instruct | torch.float32 | cpu | 16 prompts | max_new_tokens=48
batch 1  : 11.3 useful tok/s, mean latency 17.5s
best (bs=16): 38.5 useful tok/s, mean latency 11.8s, decode waste 41%
throughput gain from batching: 3.41x
